# Diccionari fonètic

Genera **només els overrides** que el TTS necessita: si una entitat ja es pronuncia bé
tal com s'escriu, no entra al diccionari i els consumidors usen la grafia original.

| `ETAPA` | conté | quan | el llegeix |
|---|---|---|---|
| `treball` | candidats de Font B | abans del round-trip | `src/verify_entities.py` |
| `final` | entitats seleccionades | després del llindar | `generate_sentences.ipynb` |

Crides, lots i prompts: `src/llm.py`. Format i cache: `src/phonetics.py`.


In [15]:
import json
import os
import sys
from pathlib import Path

ARREL = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/llm.py").exists())
sys.path.insert(0, str(ARREL / "src"))
import llm
import phonetics

IDIOMA = "Castellano"                                  # 'Catalán' | 'Euskera'
ETAPA = os.environ.get("PHONETICS_STAGE", "final")     # 'treball' | 'final'
# gpt-4.1-mini: API de pagament, sense benchmark propi. Si el cost hi arriba a
# importar, `qwen3:14b-q4_K_M` (Ollama, local i gratuït) és la primera alternativa a
# provar. Vigila en qualsevol cas el mode de fallada agressiu: un model que reescriu
# el que ja sonava bé ('Ourense' -> 'Orense') degrada activament la pronunciació, no
# és una oportunitat perduda. El que hi falta va a CORRECCIONS.
MODEL = os.environ.get("PHONETICS_MODEL", "gpt-4.1-mini")

if ETAPA not in phonetics.ETAPES:
    raise ValueError(f"ETAPA ha de ser una de {phonetics.ETAPES}, no {ETAPA!r}")

client, proveidor = llm.client_per_model(MODEL)
RUTA_TREBALL = phonetics.ruta_diccionari(IDIOMA, "treball")
RUTA_FINAL = phonetics.ruta_diccionari(IDIOMA, "final")
RUTA_CACHE = phonetics.ruta_cache(IDIOMA)
print(f"Etapa {ETAPA} | model {MODEL} ({proveidor}) | idioma {IDIOMA}")


Etapa final | model gpt-4.1-mini (openai) | idioma Castellano


In [16]:
# Overrides revisats a mà. Manen sobre el model, però només s'apliquen si l'entitat
# forma part de l'objectiu de l'etapa, així el diccionari no acumula claus alienes.
CORRECCIONS = {
    # Regles fixes confirmades sobre el TTS
    "Hamas": "Hamás",
    "FC Barcelona": "Fútbol Club Barcelona",
    "EH Bildu": "E H Bildu",
    "Spotify Camp Nou": "Espótifai Camp Nou",
    "Nico Harrison": "Nico Járrison",
    "Nico Williams": "Nico Uíliams",
    "Mohamed Shia al-Sudani": "Mohámed Shía al Sudáni",
    "Angeldahl": "Ángeldal",
    "OSCE": "Osce",
    "RTVE": "R T V E",
    "Datos RTVE": "Datos R T V E",
    "Reuters": "Róiters",
    "Steve Witkoff": "Estiv Uítcof",
    "Bruce Springsteen": "Brus Springstín",

    # Corregides arran del round-trip: l'override anterior CREAVA la fallada. La 'j'
    # castellana és una jota dura, i posar-la on hi ha una 'h' suau feia que el TTS
    # digués /xa'land/ i Whisper escrivís "Yaland" (tasa_error 1.0 fins i tot en net).
    "Erling Haaland": "Erling Hóland",
    "João Neves": "Yoáo Neves",

    # La prova A/B TTS->Whisper no va millorar aquests casos: es força la grafia crua.
    # Per repetir-la: `verify_entities.py --diccionari /dev/null`.
    "Bruno Fernandes": "Bruno Fernandes",
    "Nuno Mendes": "Nuno Mendes",
    "Pete Hegseth": "Pete Hegseth",
    "Hapoel Tel Aviv": "Hapoel Tel Aviv",
    "Washington": "Washington",
    "Airbnb": "Airbnb",
    "László Krasznahorkai": "László Krasznahorkai",
    "Dončić": "Dončić",
    "Letur": "Letur",
    "Lladró": "Lladró",
}


In [17]:
# Entitats objectiu de l'etapa.
#
# 'treball' llegeix NOMÉS Font B: `verify_entities.py` no toca mai Font A, que ja té la
# seva evidència (errors reals sobre àudio) i entra sencera sense round-trip. Font A rep
# fonètica a l'etapa 'final', quan `entidades_candidatas.json` ja té la forma definitiva.
def entitats_font_b() -> list[str]:
    d = json.loads((ARREL / "lab/entitats/es/entidades_fuente_b_validadas.json").read_text("utf-8"))
    return llm.dedupe([v["grafia_correcta"] for v in d.get("validadas", [])]
                      + [v["grafia_correcta"] for v in d.get("rechazadas", [])
                         if v["tipo"] != "NO_ENTIDAD"])


if ETAPA == "treball":
    entitats, ruta_sortida = entitats_font_b(), RUTA_TREBALL
else:
    entitats = list(json.loads(
        (ARREL / "lab/entitats/es/entidades_candidatas.json").read_text("utf-8")).keys())
    ruta_sortida = RUTA_FINAL
print(f"{len(entitats)} entitats objectiu -> {ruta_sortida.name}")

# Cache comuna a les dues etapes: hi migren els resultats ja calculats, incloses les
# decisions negatives, que no entren al diccionari però eviten tornar a consultar-les.
cache = phonetics.carregar_cache(RUTA_CACHE)
for ruta in (RUTA_TREBALL, RUTA_FINAL):
    cache.importar_diccionari(phonetics.carregar(ruta))

# Una decisió automàtica presa amb un prompt anterior no és evidència eterna; les
# revisions manuals no es toquen mai.
if n := cache.invalidar_automatiques(phonetics.POLITICA_PRONUNCIACIO_VERSION):
    print(f"  {n} decisions automàtiques antigues invalidades (política nova)")


161 entitats objectiu -> diccionari_fonetic_final_castellano.json


In [18]:
# 1) Filtre barat: només per a entitats mai vistes.
pendents = [e for e in entitats if e not in cache]
print(f"{len(entitats) - len(pendents)} decisions reutilitzades | {len(pendents)} noves")

if pendents:
    decisions, meta = llm.processar_per_lots(
        pendents, 40, client, MODEL, llm.system_decisio(IDIOMA), llm.SCHEMA_DECISIO,
        "phonetic_need_classification", "decisions", "decisio", etiqueta="Filtre")

    forcades = []
    for entitat, decisio in decisions.items():
        # Guardarraïl: una sigla, un símbol o un patró clarament estranger no es pot
        # descartar amb el classificador barat. Sense això es perdien 'FC Barcelona'
        # i 'EH Bildu', que el filtre donava per bons tal com s'escriuen.
        if decisio == "no" and phonetics.requereix_avaluacio(entitat):
            decisio = "dubte"
            forcades.append(entitat)
        if decisio == "no":
            cache.posar_sense_canvi(entitat, origen=f"llm:{MODEL}")
        else:
            cache.posar_pendent(entitat, origen=f"llm:{MODEL}:{decisio}")

    if forcades:
        print(f"  {len(forcades)} 'no' forçats a 'dubte' pel guardarraïl: "
              f"{', '.join(forcades[:8])}{'...' if len(forcades) > 8 else ''}")
    phonetics.desar_cache(cache, RUTA_CACHE, idioma=IDIOMA, model=MODEL)


161 decisions reutilitzades | 0 noves


In [19]:
# 2) El model expert només rep els positius i dubtes que encara no tenen fonètica.
pendents_fonetica = [e for e in entitats if cache.necessita_fonetica(e) and not cache.resolta(e)]
print(f"{len(pendents_fonetica)} entitats a transcriure amb {MODEL}")


def desar_lot(lot: dict) -> None:
    """Checkpoint per lot, filtrant les propostes mecànicament impossibles.
    `motiu_respelling_insegur` no jutja si la fonètica és bona: bloqueja el que no pot
    ser una reescriptura castellana (caràcters aliens, accents alterats, mots trencats)."""
    for entitat, fonetica in lot.items():
        if motiu := phonetics.motiu_respelling_insegur(entitat, fonetica):
            cache.posar_sense_canvi(entitat, origen=f"descartat_insegur:{MODEL}:{motiu}")
            cache.registre(entitat)["revisio_pendent"] = True
            print(f"  DESCARTAT insegur: {entitat!r} -> {fonetica!r} ({motiu})")
        else:
            cache.posar_fonetica(entitat, fonetica, origen=f"llm:{MODEL}")
    phonetics.desar_cache(cache, RUTA_CACHE, idioma=IDIOMA, model=MODEL)


if pendents_fonetica:
    llm.processar_per_lots(pendents_fonetica, 24, client, MODEL, llm.system_fonetica(IDIOMA),
                           llm.SCHEMA_FONETICA, "phonetic_dictionary_generation",
                           "diccionari", "transcripcio_fonetica",
                           max_tokens=4096, on_lot=desar_lot)


0 entitats a transcriure amb gpt-4.1-mini


In [20]:
# 3) Les correccions humanes manen i són compartides per les dues etapes.
for entitat in entitats:
    if (fonetica := CORRECCIONS.get(entitat)) is None:
        continue
    if phonetics.es_identitat(entitat, fonetica):
        cache.posar_sense_canvi(entitat, origen="revisio_manual")
        cache.registre(entitat)["revisat"] = True
    else:
        cache.posar_fonetica(entitat, fonetica, origen="revisio_manual", revisat=True)

# A 'final' ja es coneixen totes les grafies vives de les dues etapes; la resta són
# restes de llistes de candidats anteriors.
if ETAPA == "final" and (n := cache.podar(entitats + entitats_font_b())):
    print(f"{n} grafies obsoletes eliminades de la cache")
phonetics.desar_cache(cache, RUTA_CACHE, idioma=IDIOMA, model=MODEL)

acumulat = cache.diccionari(entitats)
if no_resoltes := [e for e in entitats if not cache.resolta(e)]:
    raise RuntimeError(f"Queden decisions sense resoldre: {no_resoltes[:10]}")

# El fitxer desat és dispers: només overrides. La cache guarda a part les decisions
# negatives perquè una reexecució no les torni a consultar.
phonetics.desar(acumulat, ruta_sortida, idioma=IDIOMA, etapa=ETAPA,
                model=MODEL, n_objectiu=len(entitats))
print(f"{len(acumulat)} overrides de {len(entitats)} entitats -> {ruta_sortida}")

if ETAPA == "treball":
    print("\nSegüent: python3 src/verify_entities.py")
else:
    pendents_rev = [g for g, e in acumulat.entrades.items() if not e.get("revisat")]
    print(f"\n{len(pendents_rev)} entrades pendents de revisió manual:")
    for g in pendents_rev:
        print(f"  {g:32} -> {acumulat.get(g)}")


31 overrides de 161 entitats -> /media/dd3/sintetic-dataset/lab/entitats/diccionaris/diccionari_fonetic_final_castellano.json

19 entrades pendents de revisió manual:
  Lamine Yamal                     -> Lamín Yamal
  Tofaş                            -> Tofash
  París FC                         -> París F C
  AlertCops                        -> Alert Cops
  Luka Dončić                      -> Luka Donchich
  Bolelli                          -> Bolélli
  Henri Duparc                     -> Enri Duparc
  Netzarim                         -> Netsarim
  Imre Kertész                     -> Imre Kértes
  Han Kang                         -> Jan Kang
  Carl Philipp                     -> Carl Filip
  Maria Theresia                   -> María Teresia
  TASIS                            -> T A S I S
  AEMET                            -> A E M E T
  DatosRTVE                        -> Datos R T V E
  Elbridge Colby                   -> Elbrídch Cóulbi
  PP                               -> P P
  An

## Revisió manual (`ETAPA='final'`)

Per corregir-ne una, posa-la a `CORRECCIONS` i torna a executar. Aquesta cel·la només
serveix per **acceptar en bloc** la resta un cop les has llegit; marcar-les fa que cap
regeneració les torni a tocar.


In [21]:
ACCEPTAR_LA_RESTA = False

if ETAPA == "final" and ACCEPTAR_LA_RESTA:
    d = phonetics.carregar(RUTA_FINAL)
    n = sum(not e.get("revisat") and not e.update({"revisat": True}) for e in d.entrades.values())
    cache.importar_diccionari(d)
    phonetics.desar_cache(cache, RUTA_CACHE, idioma=IDIOMA, model=MODEL)
    phonetics.desar(d, RUTA_FINAL, idioma=IDIOMA, etapa="final", model=MODEL)
    print(f"{n} entrades acceptades tal com estaven")
